# 05 – Einblick in die Layout-Erkennung

**Schulungs-Notebook · pp_doclayoutv3 unter dem Mikroskop**

Stufe 1 der Pipeline macht aus einer PDF-Seite eine Liste verorteter,
typisierter Blöcke in Lesereihenfolge. Im Normalbetrieb passiert das in
einem Aufruf – `Detektor.erkenne()` – und man sieht nur das Ergebnis.
Dieses Notebook zieht den Aufruf auseinander und zeigt jeden Zwischenstand.

| Teil | Inhalt |
|---|---|
| **A · Mikroskop** | *eine* Seite Schritt für Schritt: Render → 800×800-Tensor → 300 Hypothesen → Klassen → Schwelle → Masken → Lesefolge → Koordinaten → Befund |
| **B · Viewer** | Gradio-App über ganze Dokumente: Befunde über dem Original, Auffälligkeiten, Live-Modell mit Schwellenregler |

Alle Rechenschritte stammen unverändert aus `python/pdf_extraction/layout.py`;
das Zeichnen und die Sichtprüfung liegen in `einblick.py`. Dieses Notebook
**schreibt nichts** – weder Befunde noch Ausschnitte.

**Voraussetzung für Teil B:** Stufe 1 ist für mindestens ein Dokument gelaufen
(Notebook 01 oder 04). Teil A braucht nur ein PDF unter `data/raw/`.

---
## 0. Umgebung und Auswahl

Nur hier wird etwas eingestellt: welches Dokument, welche Seite, welche Schwelle.

In [ ]:
from pathlib import Path
import sys

cwd = Path.cwd().resolve()
PROJECT_ROOT = next((p for p in (cwd, *cwd.parents) if (p / "python" / "pipeline").is_dir()), None)
if PROJECT_ROOT is None:
    raise RuntimeError("Projekt-Root nicht gefunden: python/pipeline fehlt.")
if str(PROJECT_ROOT / "python") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "python"))

%load_ext autoreload
%autoreload 2

import pipeline                      # nimmt python/pdf_extraction in den Suchpfad
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

import pfade, layout, einblick, schema
from schema import PP_LABELS, Bezugsrahmen, Bbox, befund_laden

# --- Anzupassen ---------------------------------------------------------
DOK      = None     # None: erstes Dokument mit Befund, sonst erstes PDF in data/raw
SEITE    = 0        # 0-basiert wie in PyMuPDF (Seite 1 im PDF-Viewer = 0)
SCHWELLE = 0.5      # wie layout_schwelle in pipeline_config.json
# ------------------------------------------------------------------------

if DOK is None:
    mit_befund = sorted(p.name for p in pfade.BEFUNDE.glob("*") if any(p.glob("*.json")))
    DOK = (mit_befund or sorted(p.stem for p in pfade.RAW.glob("*.pdf")) or [None])[0]
if DOK is None or not pfade.quelle(DOK).exists():
    raise FileNotFoundError(f"Kein PDF unter {pfade.RAW}")
PDF = pfade.quelle(DOK)

plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False,
                     "axes.spines.right": False, "axes.edgecolor": "#8a8983",
                     "axes.titlesize": 11, "font.size": 9})

def zeige(*bilder, titel=(), breite=14, hoehe=None):
    # Mehrere Bilder nebeneinander, ohne Achsen.
    fig, achsen = plt.subplots(1, len(bilder), figsize=(breite, hoehe or breite / len(bilder) * 1.35))
    for ax, b, t in zip(np.atleast_1d(achsen), bilder, list(titel) + [""] * len(bilder)):
        ax.imshow(b, interpolation="nearest"); ax.set_title(t); ax.axis("off")
    plt.tight_layout(); plt.show()

print("Dokument:", DOK, f"({PDF.name})")
print("Seite   :", SEITE, "(0-basiert)")
print("Modell  :", pfade.ONNX_STANDARD.name)

---
# Teil A · Mikroskop

## 1. Die Seite, wie das PDF sie liefert

Ein PDF kennt keine Pixel, sondern **Punkte** (1 pt = 1/72 Zoll). Für die
Layout-Analyse wird die Seite mit **200 dpi** gerastert
(`layout.RENDER_DPI`). Aus einer Seite von 547 × 737 pt werden so
547 · 200/72 ≈ 1520 × 2048 Pixel.

In [ ]:
bild, pg = layout.seite_rendern(PDF, layout.RENDER_DPI, SEITE)
H, W = bild.shape[:2]
print(f"PDF-Seite : {pg.rect.width:.1f} × {pg.rect.height:.1f} pt")
print(f"Render    : {W} × {H} px bei {layout.RENDER_DPI} dpi  (Faktor {layout.RENDER_DPI/72:.3f} px/pt)")
print(f"Textlayer : {len(pg.get_text())} Zeichen – das Modell benutzt ihn NICHT, es sieht nur Pixel.")
zeige(bild, titel=[f"Original-Render {W}×{H}"], breite=7, hoehe=9.5)

## 2. Was das Modell sieht: ein 800 × 800-Tensor

`layout.vorverarbeiten()` macht drei Dinge – genau so, wie PaddleX das Modell
trainiert hat:

1. **Skalieren auf 800 × 800** mit `cv2.INTER_CUBIC` – und zwar **ohne das
   Seitenverhältnis zu erhalten** (`keep_ratio=false`). Die Hochformatseite wird
   seitlich gestreckt bzw. in der Höhe gestaucht.
2. **Werte auf 0…1** (`/255`), keine ImageNet-Normierung.
3. **HWC → NCHW**: aus (Höhe, Breite, Kanal) wird (Batch, Kanal, Höhe, Breite).

Die Verzerrung ist kein Fehler: Das Modell gibt Boxen **normiert auf 0…1**
aus. Multipliziert man sie mit Breite und Höhe des Originals, ist die Streckung
wieder aufgehoben – achsenweise.

In [ ]:
tensor = layout.vorverarbeiten(bild)
modellbild = einblick.tensor_als_bild(tensor)
print("Tensor  :", tensor.shape, tensor.dtype, f"Werte {tensor.min():.2f} … {tensor.max():.2f}")
print(f"Streckung: x × {800/W:.3f}, y × {800/H:.3f}  → Seitenverhältnis {W/H:.2f} wird zu 1.00")
zeige(bild, modellbild, titel=[f"Original {W}×{H}", "Modelleingang 800×800 (verzerrt!)"], breite=13, hoehe=8)

**Wie viel Auflösung bleibt übrig?** Derselbe Ausschnitt einmal aus dem
Original, einmal aus dem Modelleingang – beide auf dieselbe Breite gezogen,
ohne Glättung, damit man die echten Pixel sieht. Der Text schrumpft auf rund
ein Viertel der Pixel und wird in der Höhe stärker gestaucht als in der
Breite. Das Modell muss ihn aber gar nicht lesen: Es erkennt *Formen* –
Zeilen, Spalten, Weißraum, Bildflächen, Schriftgröße. Den Text liest erst
Stufe 2 (PaddleOCR-VL) aus einem hochaufgelösten Ausschnitt (300 dpi).

In [ ]:
# Relativer Ausschnitt (x0, y0, x1, y1) in Anteilen der Seite – gern ändern.
REL = (0.35, 0.60, 0.95, 0.75)
o = bild[int(REL[1]*H):int(REL[3]*H), int(REL[0]*W):int(REL[2]*W)]
m = modellbild[int(REL[1]*800):int(REL[3]*800), int(REL[0]*800):int(REL[2]*800)]
zeige(o, m, titel=[f"Original {o.shape[1]}×{o.shape[0]} px", f"Modell {m.shape[1]}×{m.shape[0]} px"], breite=14, hoehe=3.5)

## 3. Ein Vorwärtslauf – vier Köpfe

pp_doclayoutv3 ist ein **DETR-artiger Detektor** (RT-DETR-Familie): ein
CNN-Backbone erzeugt Merkmalskarten, ein Transformer-Encoder verdichtet sie,
und ein Decoder lässt **300 Object Queries** – gelernte „Fragen“ – auf das Bild
schauen. Jede Query liefert am Ende genau eine Hypothese.

Der ONNX-Export gibt die Rohköpfe unverändert heraus; alles Weitere passiert
in `layout.dekodieren()` – also bei uns, sichtbar.

In [ ]:
detektor = layout.Detektor(pfade.ONNX_STANDARD)
detektor.signatur_zeigen()

lauf = einblick.roh_rechnen(detektor, PDF, SEITE)
print(f"\nInferenz: {lauf.dauer_s:.2f} s auf der CPU\n")

BEDEUTUNG = {
    "logits":       "je Query ein Wert je Klasse (25) – vor der Sigmoid",
    "pred_boxes":   "je Query eine Box (cx, cy, w, h), normiert auf 0…1",
    "order_logits": "Query × Query: 'kommt j nach i?' – die Lesefolge",
    "out_masks":    "je Query eine 200×200-Maske (Stride 4 auf 800×800)",
}
for name, arr in lauf.ausgaben.items():
    print(f"  {name:13s} {str(arr.shape):22s} {BEDEUTUNG.get(name, '')}")

## 4. 300 Hypothesen – und warum es kein NMS braucht

Klassische Detektoren (YOLO, Faster R-CNN) erzeugen tausende überlappende
Kandidaten und räumen sie mit *Non-Maximum Suppression* auf. DETR wird anders
trainiert: Beim Training wird jede echte Box per **Hungarian Matching** genau
*einer* Query zugeordnet; alle übrigen Queries lernen „kein Objekt“. Das
Ergebnis sieht man unten:

* **links** alle 300 Boxen auf dem Modelleingang – Deckkraft ∝ bester
  Klassenscore. Grau = unter der Schwelle, farbig = darüber.
* **rechts** die Verteilung der besten Scores. Wenige Queries sind sich sehr
  sicher, die große Masse liegt nahe 0. Dazwischen: die interessanten Grenzfälle.

In [ ]:
p = lauf.wahrscheinlichkeiten                 # (300, 25)
beste = p.max(axis=1)

fig, (a1, a2) = plt.subplots(1, 2, figsize=(14, 6.5), gridspec_kw={"width_ratios": [1, 1.1]})
a1.imshow(einblick.zeichne_queries(modellbild, lauf, SCHWELLE)); a1.axis("off")
a1.set_title("Alle 300 Queries auf dem 800×800-Eingang")
a2.hist(beste, bins=np.linspace(0, 1, 41), color="#2a78d6", edgecolor="white", linewidth=1)
a2.axvline(SCHWELLE, color="#0b0b0b", lw=1.5, ls="--")
a2.text(SCHWELLE + 0.01, a2.get_ylim()[1] * 0.9, f"Schwelle {SCHWELLE}", fontsize=9)
a2.set_yscale("log"); a2.set_xlabel("bester Klassenscore je Query"); a2.set_ylabel("Anzahl Queries (log)")
a2.set_title("Wie sicher ist sich jede Query?"); a2.grid(axis="y", color="#e8e7e2", lw=0.8)
plt.tight_layout(); plt.show()

for s in (0.1, 0.3, 0.5, 0.7, 0.9):
    print(f"  Queries mit Score ≥ {s:.1f}: {(beste >= s).sum():3d}")

## 5. Welche Klasse? – die Klassen-Heatmap

Jede Query hat 25 Klassen-Logits. Anders als bei einem Klassifikator werden
sie **nicht per Softmax** gegeneinander normiert, sondern jede einzeln per
**Sigmoid** (Focal Loss). Eine Query kann also für zwei Klassen gleichzeitig
hohe Werte haben – oder für keine.

`dekodieren()` nimmt deshalb die Top-300 über das **ganze Gitter Query × Klasse**.
Eine Query mit zwei starken Klassen erzeugt **zwei Blöcke** an derselben Stelle.
(Die Lesekante zwischen solchen Zwillingen wird übersprungen, siehe `layout.py`.)

**Worauf achten:** Queries knapp unter der Schwelle, deren Masse sich auf zwei
verwandte Klassen verteilt. Auf der Beispielseite ist das die Bildunterschrift
„Abb. 9.3“ oben links: je etwa 0.45 für `figure_title` *und* `vision_footnote`.
Keine der beiden Klassen erreicht 0.5 – ohne Gegenmaßnahme fällt das Element
ganz heraus, obwohl das Modell es klar „gesehen“ hat.

**Gegenmaßnahme: die Familienrettung.** `dekodieren()` macht deshalb einen
zweiten Durchgang über die noch nicht gewählten Queries. Für Klassen, zwischen
denen das Modell erfahrungsgemäß schwankt (`layout.RETTUNGS_FAMILIEN`:
Bildbezug, Überschriften, Formeln), wird gefragt, ob *mindestens eine* davon
zutrifft:

$$p_\text{Familie} = 1 - \prod_k (1 - p_k) \qquad 1 - (1-0.45)^2 \approx 0.70$$

Reicht das für die Schwelle, ist die beste Einzelklasse ≥ 0.3 und liegt die Box
nicht schon in einem angenommenen Block, wird die Query aufgenommen – mit der
besten Einzelklasse als Label und dem Vermerk `familien_score`. Einfach die
Schwelle zu senken wäre schlechter: Bei 0.3 entstehen **Zwillinge** (dieselbe
Query als zwei Blöcke).

In [ ]:
N = 30
top = np.argsort(-beste)[:N]
fig, ax = plt.subplots(figsize=(14, 8.5))
im = ax.imshow(p[top], cmap="Blues", vmin=0, vmax=1, aspect="auto")
ax.set_xticks(range(len(PP_LABELS)), PP_LABELS, rotation=60, ha="right")
ax.set_yticks(range(N), [f"Q{q:03d}" for q in top])
for i, q in enumerate(top):                     # Zahl nur dort, wo es etwas zu sagen gibt
    for k in np.where(p[q] >= 0.2)[0]:
        ax.text(k, i, f"{p[q, k]:.2f}", ha="center", va="center", fontsize=7,
                color="white" if p[q, k] > 0.6 else "#0b0b0b")
ax.set_title(f"Sigmoid-Wahrscheinlichkeit je Klasse – die {N} sichersten Queries")
fig.colorbar(im, ax=ax, fraction=0.025, pad=0.01)
plt.tight_layout(); plt.show()

ohne, _ = lauf.dekodieren(SCHWELLE, familien_retten=False)
mit, _ = lauf.dekodieren(SCHWELLE, familien_retten=True)
print(f"Schwelle {SCHWELLE}: {len(ohne)} Blöcke ohne, {len(mit)} mit Familienrettung")
for b in mit:
    if b.familien_score is not None:
        print(f"  gerettet: Q{b.query_id:03d} als {b.pp_label} – Einzelklasse {b.score:.2f}, "
              f"Familie {b.familien_score:.2f}")
print()
print("Grenzfälle – zweitbeste Klasse ≥ 0.2:")
for q in top:
    k1, k2 = np.argsort(-p[q])[:2]
    if p[q, k2] >= 0.2:
        print(f"  Q{q:03d}: {PP_LABELS[k1]} {p[q,k1]:.2f}  vs.  {PP_LABELS[k2]} {p[q,k2]:.2f}")

## 6. Die Schwelle – interaktiv

Die Schwelle ist der wichtigste Regler der Stufe 1. Zu hoch: Elemente fehlen
(achte auf die Bildunterschrift oben links auf der Beispielseite!). Zu niedrig:
Dubletten und Rauschen. Der Regler dekodiert nur neu – das Modell rechnet
nicht noch einmal. Mit dem Häkchen lässt sich die Familienrettung ein- und
ausschalten.

In [ ]:
def schwelle_zeigen(schwelle=SCHWELLE, familien_retten=True):
    bloecke, kanten = lauf.dekodieren(schwelle, familien_retten=familien_retten)
    ov = einblick.zeichne_bloecke(bild, bloecke, 1.0, kanten=kanten)
    fig, ax = plt.subplots(figsize=(9, 12))
    ax.imshow(ov); ax.axis("off")
    ax.set_title(f"Schwelle {schwelle:.2f}: {len(bloecke)} Blöcke")
    plt.tight_layout(); plt.show()

try:
    import ipywidgets as w
    w.interact(schwelle_zeigen, schwelle=w.FloatSlider(
        value=SCHWELLE, min=0.05, max=0.95, step=0.05, continuous_update=False,
        description="Schwelle"), familien_retten=w.Checkbox(True, description="Familienrettung"))
except ImportError:
    for s in (0.3, SCHWELLE, 0.7):
        schwelle_zeigen(s)

## 7. Masken → Polygone

Der vierte Kopf liefert je Query eine **Instanzmaske** mit 200 × 200 Pixeln –
ein Viertel der Eingangsauflösung (Stride 4), also ebenfalls im verzerrten
800er-Raum. `polygone_ziehen()` schneidet die Maske auf die Box zu, skaliert
sie auf Bildpixel, sucht die Außenkontur (`cv2.findContours`) und vereinfacht
sie (`approxPolyDP`). So entstehen Polygone, die z. B. Text **um ein Bild
herum** fassen können – ein Rechteck könnte das nicht.

In [ ]:
bloecke, kanten = lauf.dekodieren(SCHWELLE)
masken = layout._sigmoid(lauf.ausgaben["out_masks"][0])      # (300, 200, 200)

zeigen = bloecke[:12]
fig, achsen = plt.subplots(2, 6, figsize=(14, 5.4))
for ax, b in zip(achsen.flat, zeigen):
    ax.imshow(masken[b.query_id], cmap="Blues", vmin=0, vmax=1)
    ax.set_title(f"#{b.id} {b.pp_label}\nQ{b.query_id}", fontsize=8); ax.axis("off")
for ax in list(achsen.flat)[len(zeigen):]:
    ax.axis("off")
fig.suptitle("Rohmasken (200×200, Sigmoid) der ersten Blöcke in Lesefolge", fontsize=11)
plt.tight_layout(); plt.show()

In [ ]:
# Der Block mit den meisten Polygonecken: Rechteck gegen Polygon
b = max(bloecke, key=lambda b: len(b.polygon or []))
x0, y0, x1, y1 = (int(v) for v in (b.bbox.x0, b.bbox.y0, b.bbox.x1, b.bbox.y1))
rand = 40
ys, xs = slice(max(0, y0 - rand), y1 + rand), slice(max(0, x0 - rand), x1 + rand)

als_box = einblick.zeichne_bloecke(bild, [b], polygone=False, lesefolge=False)
als_poly = einblick.zeichne_bloecke(bild, [b], polygone=True, lesefolge=False)
maske_gross = np.array(einblick._als_pil((masken[b.query_id] * 255).astype(np.uint8)).resize((W, H)))
print(f"#{b.id} {b.pp_label}: Polygon mit {len(b.polygon or [])} Ecken")
zeige(np.asarray(als_box)[ys, xs], maske_gross[ys, xs], np.asarray(als_poly)[ys, xs],
      titel=["Bounding Box", "Maske (auf Bildgröße gestreckt)", "Polygon aus der Maske"],
      breite=15, hoehe=4.5)

## 8. Lesereihenfolge aus der Zeigermatrix

Der Kopf `order_logits` (300 × 300) beantwortet für jedes Query-Paar die Frage
*„kommt j nach i?“*. `lese_raenge()` zählt für jede Query die Stimmen „ich
komme später“ – je mehr, desto höher der Rang. Sortiert man die Blöcke nach
diesem Rang, sollte die Matrix unten **oberhalb der Diagonale blau (1)** und
**unterhalb orange (0)** sein. Jede Zelle, die davon abweicht, ist eine Stelle,
an der das Modell unsicher ist oder sich widerspricht.

Die Wahrscheinlichkeit sättigt schnell auf 0 oder 1; deshalb speichert der
Befund zusätzlich die **Logit-Marge** jeder Kante. Negative Marge = das Modell
widerspricht der eigenen Rangfolge – im Bild rot.

In [ ]:
ids = [b.query_id for b in bloecke]
P = einblick.folgematrix(lauf, ids)
divergent = LinearSegmentedColormap.from_list("folge", ["#eb6834", "#e8e7e2", "#2a78d6"])

fig, (a1, a2) = plt.subplots(1, 2, figsize=(15, 8), gridspec_kw={"width_ratios": [1, 1]})
im = a1.imshow(P, cmap=divergent, vmin=0, vmax=1)
namen = [f"#{b.id} {b.pp_label[:10]}" for b in bloecke]
a1.set_xticks(range(len(namen)), namen, rotation=90, fontsize=7)
a1.set_yticks(range(len(namen)), namen, fontsize=7)
a1.set_xlabel("j"); a1.set_ylabel("i"); a1.set_title("P(j kommt nach i), Blöcke in Lesefolge")
fig.colorbar(im, ax=a1, fraction=0.04, pad=0.02)
a2.imshow(einblick.zeichne_bloecke(bild, bloecke, kanten=kanten, beschriftung=False)); a2.axis("off")
a2.set_title("Lesefolge auf der Seite (rot = Widerspruch)")
plt.tight_layout(); plt.show()

from schema import SeitenBefund, Stufe
tmp = SeitenBefund(quelle_datei=PDF.name, seite=SEITE, seite_breite_pt=pg.rect.width,
                   seite_hoehe_pt=pg.rect.height, render_dpi=lauf.dpi, bild_breite_px=W,
                   bild_hoehe_px=H, bloecke=bloecke, kanten=kanten, stufe=Stufe.LAYOUT)
print("Die fünf unsichersten Übergänge (nach Marge):")
for k in tmp.schwaechste_kanten(5):
    print(f"  #{k.von:2d} {bloecke[k.von].pp_label:16s} → #{k.nach:2d} {bloecke[k.nach].pp_label:16s} "
          f"Marge {k.marge:8.1f}  P={k.konfidenz:.3f}")

## 9. Koordinaten: vom Tensor bis in den PDF-Textlayer

Eine Zahl wie `400` ist ohne Bezugsrahmen bedeutungslos. Das Schema
(`schema.Bezugsrahmen`) kennt vier Rahmen; hier läuft **eine** Box durch alle.
Am Ende steht ein Rechteck in PDF-Punkten – damit lässt sich der eingebettete
Textlayer des PDFs genau an dieser Stelle auslesen. Das ist die Probe aufs
Exempel: Stimmt der Text, stimmt die ganze Umrechnungskette.

In [ ]:
b = max((b for b in bloecke if b.pp_label == "text"), key=lambda b: b.bbox.flaeche, default=bloecke[0])
cx, cy, bw, bh = lauf.ausgaben["pred_boxes"][0][b.query_id]

m800 = Bbox.aus_cxcywh(cx * 800, cy * 800, bw * 800, bh * 800, rahmen=Bezugsrahmen.MODELL_800)
px = schema.zurueck_ins_bild(m800, W, H)
pt = schema.ins_pdf(px, lauf.dpi)

print(f"Block #{b.id} {b.pp_label} (Query {b.query_id})\n")
print(f"  Modellausgabe  cxcywh 0…1 : {cx:.4f} {cy:.4f} {bw:.4f} {bh:.4f}")
print(f"  MODELL_800     xyxy       : {m800.x0:7.1f} {m800.y0:7.1f} {m800.x1:7.1f} {m800.y1:7.1f}")
print(f"  BILD_PIXEL     @{lauf.dpi} dpi  : {px.x0:7.1f} {px.y0:7.1f} {px.x1:7.1f} {px.y1:7.1f}   (Befund: {b.bbox.x0:.1f} {b.bbox.y0:.1f} …)")
print(f"  SEITE_PUNKT    1/72 Zoll  : {pt.x0:7.1f} {pt.y0:7.1f} {pt.x1:7.1f} {pt.y1:7.1f}")
print(f"  Gemma 0…1000   [y0,x0,y1,x1]: {schema.nach_gemma(px, W, H)}")

import pymupdf
text = pg.get_textbox(pymupdf.Rect(pt.x0, pt.y0, pt.x1, pt.y1)).strip()
print("\nPDF-Textlayer in diesem Rechteck:\n")
print("  " + (text[:400].replace("\n", "\n  ") if text else "(kein Textlayer – gescannte Seite?)"))

## 10. Das Ergebnis: der Befund

`dekodieren()` fasst alles zusammen: Blöcke mit Klasse, Score, Box, Polygon und
Lese-Index, dazu die Kanten. `Detektor.erkenne()` packt das in einen
`SeitenBefund` und die Pipeline speichert ihn als JSON.

Liegt schon ein gespeicherter Befund vor, lohnt der Vergleich: **nach Stufe 2
sind oft weniger Blöcke da** – `erkennung.gruppieren()` hat nebeneinander
liegende Textblöcke zusammengeführt, und die ids sind neu vergeben. Weil
Stufe 2 die Datei der Stufe 1 überschreibt, verweist der Befund auf die
Mitglieder über ihre **Query-ids** (`zusammengefuehrt_queries`, Vermerk in
`warnungen`) – die sind über alle Stufen stabil und führen direkt zurück in
die Rohausgabe oben.

In [ ]:
gespeichert = pfade.befund(DOK, SEITE)
if gespeichert.exists():
    bf = befund_laden(gespeichert)
    print(f"Live, Schwelle {SCHWELLE}: {len(bloecke)} Blöcke")
    print(f"Gespeichert      : {len(bf.bloecke)} Blöcke, Stufe {bf.stufe.value}")
    for w_ in bf.warnungen:
        print("   ·", w_)
    zeige(einblick.zeichne_bloecke(bild, bloecke, kanten=kanten),
          einblick.zeichne_befund(bild, bf),
          titel=["Live: Stufe 1", f"Gespeichert: Stufe {bf.stufe.value}"], breite=15, hoehe=10.5)
    print("\nAuffälligkeiten des gespeicherten Befunds:")
    for schwere, text_ in einblick.auffaelligkeiten(bf):
        print(f"  [{schwere:7s}] {text_}")
else:
    print("Noch kein gespeicherter Befund – nur das Live-Ergebnis:")
    zeige(einblick.zeichne_bloecke(bild, bloecke, kanten=kanten), breite=8, hoehe=11)

In [ ]:
# So sieht ein einzelner Block im JSON aus
print(bloecke[0].model_dump_json(indent=2)[:1200])

## 11. Was man hier *nicht* sieht

* **Attention-Maps** des Encoders und Decoders: Der ONNX-Export gibt nur die
  vier Köpfe heraus. Um zu sehen, *wohin* eine Query schaut, bräuchte es einen
  eigenen Export mit zusätzlichen Ausgängen.
* **Backbone-Merkmale**: dito – die Zwischenschichten sind im Graphen, aber
  nicht als Ausgang.
* **Warum** eine Klasse gewählt wurde: Die Heatmap zeigt *was* das Modell
  glaubt, nicht weshalb. Für die Praxis reicht meist die Frage, *wo* es
  unsicher ist – und genau die beantworten Score, zweitbeste Klasse und Marge.

### Übungen

1. `SCHWELLE` auf 0.3 setzen: Welche Elemente kommen dazu, welche davon sind
   sinnvoll, welche Dubletten?
2. Die Marginalien der Beispielseite sind als `text` erkannt, nicht als
   `aside_text`. In welchen Strom geraten sie damit, und was bedeutet das für
   die Lesefolge im Markdown?
3. Das zweite PDF in `data/raw` (Formeln!) in Zelle 0 als `DOK` wählen. Wie
   verteilen sich `display_formula` und `inline_formula`?

---
# Teil B · Layout-Viewer

Eine Gradio-App über **alle Dokumente** in `data/raw`:

| Element | Funktion |
|---|---|
| **Dokument / Seite / ◀ ▶** | blättern; Seitenzahlen hier 1-basiert wie im PDF-Viewer |
| **Auffällige Seiten** | Seiten mit Befund, nach `einblick.verdachtswert` sortiert – für die Sichtprüfung großer Bücher |
| **Quelle** | *Befund (JSON)*: gespeicherter Stand (auch nach Stufe 2) · *Live-Modell*: frisch gerechnet, mit **Schwellenregler** |
| **Färbung** | nach Klassenfamilie oder nach Lesestrom (haupt / marginalie / boilerplate / apparat) |
| **Klick auf eine Box** oder Tabellenzeile | Klasse, Score, Query, Lese-Index, Kanten mit Marge, erkannter Text, Ausschnitt |

Auffälligkeiten (⛔ Fehler, ⚠️ Warnung, ℹ️ Hinweis): übersprungene Seite, Score
unter 0.6, Box in Box, Lesekante mit negativer Marge, textartiger Block ohne
Text, schmaler Fließtext in der Randspalte (Marginalie verkannt?).
Die Regeln stehen in `einblick.auffaelligkeiten` – bewusst einfach, sie
sortieren vor und entscheiden nichts.

Die App öffnet sich im Browser. `prevent_thread_lock=True` gibt den Kernel
sofort wieder frei; die Zelle darunter beendet sie.

In [ ]:
from pipeline import layout_viewer
viewer = layout_viewer.starten(inbrowser=True, prevent_thread_lock=True)

In [ ]:
viewer.close()

---
## HTML-Bericht zum Verschicken

Für alle, die keine Python-Umgebung haben: `einblick_html.exportieren()`
schreibt **eine** HTML-Datei nach `data/interim/kontrolle/<dok>_layout.html`.
Seitenbilder (JPEG) und Befunde sind eingebettet, die Overlays zeichnet der
Browser – Blättern (auch mit ← →), Filtern, Klick auf Boxen und
Auffälligkeiten funktionieren offline.

Größe: rund 170 KB je Seite bei 100 dpi. Für ein dickes Buch
`nur_auffaellige=True`, eine Seitenauswahl (`seiten=[…]`, 0-basiert) oder ein
kleineres `dpi`. Im Viewer gibt es denselben Export als Knopf.

**Achtung beim Weitergeben:** Die Datei enthält die Seitenbilder und den
erkannten Text – also den Inhalt des Buchs.

In [ ]:
import einblick_html
bericht = einblick_html.exportieren(DOK, nur_auffaellige=False, dpi=100)